In [1]:
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

import time
import json
from datetime import datetime

In [2]:
TARGET_URL = "https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2"
# Đặt tên file output dựa trên thời gian chạy để tránh ghi đè
TIMESTAMP_STR = datetime.now().strftime('%Y%m%d_%H%M%S')
HTML_OUTPUT_FILENAME = f"topcv_selenium_page_{TIMESTAMP_STR}.html"
JSON_OUTPUT_FILENAME = f"topcv_jobs_data_{TIMESTAMP_STR}.json"

print(f"URL Mục tiêu: {TARGET_URL}")
print(f"File HTML sẽ được lưu là: {HTML_OUTPUT_FILENAME}")
print(f"File JSON sẽ được lưu là: {JSON_OUTPUT_FILENAME}")

URL Mục tiêu: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
File HTML sẽ được lưu là: topcv_selenium_page_20250521_150047.html
File JSON sẽ được lưu là: topcv_jobs_data_20250521_150047.json


# Cell 2: Configure and Initialize Selenium WebDriver


In [3]:

print("Đang cấu hình Chrome Options ........")
chrome_options = Options()

# --- Các tùy chọn quan trọng ---
# 1. Chạy ở chế độ "headless" (không mở cửa sổ trình duyệt vật lý)
#    Khi debug, bạn nên comment dòng này lại để có thể thấy trình duyệt hoạt động.
# chrome_options.add_argument("--headless") 

# 2. Các cờ thường cần thiết khi chạy trong môi trường Linux hoặc container (như Docker)
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage") # Khắc phục lỗi liên quan đến /dev/shm khi thiếu tài nguyên

# 3. Tắt GPU, thường không cần thiết cho scraping và có thể cải thiện hiệu suất/ổn định
chrome_options.add_argument("--disable-gpu")

# 4. Đặt User-Agent giống như trình duyệt thông thường để tránh bị phát hiện là bot
#    Bạn có thể thay bằng User-Agent từ trình duyệt của mình nếu muốn
chrome_options.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36")

# 5. Đặt kích thước cửa sổ (có thể quan trọng đối với một số trang web responsive)
chrome_options.add_argument("--window-size=1920,1080")

# 6. (Tùy chọn) Chạy ở chế độ ẩn danh
# chrome_options.add_argument("--incognito")

# 7. (Tùy chọn) Tắt thông báo "Chrome is being controlled by automated test software"
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)

# 8. (Tùy chọn) Tắt hình ảnh để tải trang nhanh hơn (nếu không cần hình ảnh)
# chrome_options.add_argument('--blink-settings=imagesEnabled=false')

print("Chrome Options đã được cấu hình.")

# --- Khởi tạo WebDriver ---
# Selenium 4 sẽ tự động tìm chromedriver nếu nó nằm trong PATH hệ thống.
# Nếu bạn đặt chromedriver ở một vị trí cụ thể và không có trong PATH, bạn cần dùng Service:
# service = Service(executable_path='/duong/dan/toi/chromedriver')
# driver = webdriver.Chrome(service=service, options=chrome_options)

try:
    print("Đang khởi tạo WebDriver...")
    driver = webdriver.Chrome(options=chrome_options)
    print("WebDriver đã được khởi tạo thành công!")
except Exception as e:
    print(f"Lỗi khi khởi tạo WebDriver: {e}")
    print("Hãy đảm bảo ChromeDriver đã được cài đặt đúng và phiên bản tương thích với Chrome của bạn.")
    print("Nếu ChromeDriver không nằm trong PATH, bạn cần chỉ định 'executable_path' thông qua 'Service'.")
    driver = None # Đặt driver là None nếu khởi tạo lỗi

# Để đảm bảo driver được đóng nếu notebook bị ngắt đột ngột hoặc cell này chạy lại nhiều lần
# chúng ta sẽ lưu trữ driver vào một biến global (hơi xấu nhưng tiện cho notebook)
# hoặc tốt hơn là quản lý nó cẩn thận trong các cell sau.
# Hiện tại, chúng ta sẽ đóng nó ở cell cuối cùng.

Đang cấu hình Chrome Options ........
Chrome Options đã được cấu hình.
Đang khởi tạo WebDriver...
WebDriver đã được khởi tạo thành công!


# Cell 3: Navigate to Target URL, Wait, and Get Page Source


In [4]:
#Kiểm tra xem biến drivẻ từ cell trước có tồn tại và không phải là None không

if 'driver' in locals() and driver is not None:
    try:
        print(f"Đang điều khiển trình duyệt truy cập URL: {TARGET_URL}")
        driver.get(TARGET_URL) # Lệnh để trình duyệt mở URL
        print(f"Đã gửi yêu cầu tải trang. URL trên trình duyệt tên là: {driver.current_url}")
        
        # Chờ một khoảng thời gian cố định để trang tải hoàn chỉnh (bao gồm JavaScript)
        # Đối với các trang phức tạp, ta có thể cần phải tăng thời gian này
        # hoặc sử dụng WebDriverWait để đợi một element cụ thể xuất hiện.
        wait_time_seconds = 14 # Thử với 14 giây, có thể tăng nếu cần
        print(f"Đang đợi {wait_time_seconds} giây để nội dung trang tải hoàn chỉnh...")
        time.sleep(wait_time_seconds)
        print("Thời gian chờ đã kết thúc.")
        
        # Lấy mã nguồn HTML của trang sau khi đã tải và JavaScript (nếu có) đã chạy
        print("Đang lấy mã nguồn HTML của trang (page_source)...")
        page_source_html = driver.page_source
        
        if page_source_html:
            print(f"Đã lấy được page_source HTML (kích thước: {len(page_source_html)} bytes).")
            
            # Lưu nội dung HTML ra file để kiểm tra và tìm selector sau này
            with open(HTML_OUTPUT_FILENAME, "w", encoding="utf-8") as f:
                f.write(page_source_html)
            print(f"Nội dung HTML của trang TopCV đã được lưu vào file: '{HTML_OUTPUT_FILENAME}'")
            print("Hãy mở file này bằng trình duyệt để kiểm tra nội dung và chuẩn bị cho việc tìm selector.")
        else:
            print("Không lấy được page_source HTML. Trang có thể chưa tải xong hoặc có lỗi.")
    except Exception as e:
        print(f"Lỗi trong quá trình truy cập URL hoặc lấy page_source: {e}")
        # Cân nhắc đóng driver nếu có lỗi nghiêm trọng ở đây, 
        # nhưng chúng ta sẽ đóng ở cell cuối cùng để đảm bảo.
else:
    print("Biến 'driver' không tồn tại hoặc chưa được khởi tạo. Hãy chạy lại Cell 2.")

Đang điều khiển trình duyệt truy cập URL: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
Đã gửi yêu cầu tải trang. URL trên trình duyệt tên là: https://www.topcv.vn/tim-viec-lam-moi-nhat-tai-ho-chi-minh-l2?exp=1&type_keyword=0&sba=1&locations=l2
Đang đợi 14 giây để nội dung trang tải hoàn chỉnh...
Thời gian chờ đã kết thúc.
Đang lấy mã nguồn HTML của trang (page_source)...
Đã lấy được page_source HTML (kích thước: 1440994 bytes).
Nội dung HTML của trang TopCV đã được lưu vào file: 'topcv_selenium_page_20250521_150047.html'
Hãy mở file này bằng trình duyệt để kiểm tra nội dung và chuẩn bị cho việc tìm selector.


# Cell 4: Parse HTML with BeautifulSoup and Extract Data

In [5]:
if 'page_source_html' in locals() and page_source_html is not None:
    print(f"Đang thực hiện phân tích (Parse) HTML với (kích thước là: {len(page_source_html)} bytes) bằng BeautifulSoup...")
    soup = BeautifulSoup(page_source_html, 'html.parser')
    
    all_jobs_data = [] 
    
    job_list_container = soup.find('div', class_='box-job-list')
        
    if job_list_container:
        print(f"Đã tìm thấy container 'div.box-job-list'.")
        job_item_elements = job_list_container.find_all('div', class_='job-item-search-result')
        print(f"Tìm thấy tổng cộng {len(job_item_elements)} job items trong 'div.box-job-list'.")
        
        if not job_item_elements:
             print("!!! Cảnh báo: Tìm thấy 'div.box-job-list' nhưng không có 'div.job-item-search-result' nào bên trong.")

        for index, item_element in enumerate(job_item_elements):
            print(f"\n --- Đang xử lý job item thứ {index + 1}/{len(job_item_elements)} ---")
            job_data = {
                'job_title': None, 'job_url': None, 'company_name': None,
                'salary': None, 'location': None, 'experience': None, 'post_date': None,
            }
            
            # --- Tên công việc (Job Title) và URL chi tiết ---
            # Giả sử title_block là một div có class 'title-block' (hoặc tương tự) chứa h3.title
            title_block_div = item_element.find('div', class_='title-block') # Kiểm tra lại class này
            if title_block_div:
                title_h3 = title_block_div.find('h3', class_=['title', 'title highlight']) # Có thể có hoặc không có 'highlight'
                if title_h3:
                    title_tag_a = title_h3.find('a')
                    if title_tag_a:
                        # Job title có thể nằm trong span bên trong a, hoặc text trực tiếp của a
                        job_title_span = title_tag_a.find('span') 
                        if job_title_span:
                            job_data['job_title'] = job_title_span.get_text(strip=True)
                        else: 
                            job_data['job_title'] = title_tag_a.get_text(strip=True) 
                        
                        job_data['job_url'] = title_tag_a.get('href')
                        if job_data['job_url'] and job_data['job_url'].startswith('/'):
                            job_data['job_url'] = "https://www.topcv.vn" + job_data['job_url']
            
            if job_data['job_title']: print(f"  Tên công việc (Job title) là: {job_data['job_title']}")
            else: print("  Không tìm thấy tên công việc (Job title).")
            if job_data['job_url']: print(f"  URL chi tiết công việc là: {job_data['job_url']}")
            # else: print("  Không tìm thấy URL chi tiết công việc.") # Không cần thiết nếu title không có

            # --- Tên công ty (Company Name) ---
            # Thử tìm thẻ a chứa tên công ty trực tiếp từ item_element
            company_tag_a = item_element.find('a', class_=['company', 'company job-pro'])
            if company_tag_a:
                company_name_span = company_tag_a.find('span', class_='company-name')
                if company_name_span:
                    job_data['company_name'] = company_name_span.get_text(strip=True)
                        
            if job_data['company_name']: print(f"  Tên công ty (Company name) là: {job_data['company_name']}")
            else: print("  Không tìm thấy tên công ty (Company name) nào.")    
            
            # === Mức lương (salary) ===
            salary_tag = item_element.find(['span', 'div', 'p', 'label'], class_='title-salary')
            if salary_tag:
                job_data['salary'] = salary_tag.get_text(strip=True).replace('\n', '').replace('\r', '').strip()
            
            if job_data['salary']: print(f"  Mức lương (Salary) là: {job_data['salary']}")
            else: print("  Không tìm thấy mức lương (Salary).")
            
            # === Địa chỉ (location) ===
            location_label = item_element.find('label', class_='address truncate')
            if location_label:
                location_span = location_label.find('span', class_='city-text')
                if location_span:
                    job_data['location'] = location_span.get_text(strip=True)
                    
            if job_data['location']: print(f"  Địa chỉ (Location) là: {job_data['location']}")
            else: print("  Không tìm thấy địa chỉ (Location).")
                
            # === Kinh nghiệm (experience) ===
            exp_label = item_element.find('label', class_='exp')
            if exp_label:
                exp_span = exp_label.find('span')
                if exp_span:
                    job_data['experience'] = exp_span.get_text(strip=True)
            
            if job_data['experience']: print(f"  Kinh nghiệm (Experience) là: {job_data['experience']}")
            else: print("  Không tìm thấy kinh nghiệm (Experience).")
                
            # === Ngày đăng (post_date) ===
            date_tag = item_element.find('label', class_='address mobile-hidden label-update')
            if date_tag:
                raw_date_text = date_tag.get_text(strip=True)
                job_data['post_date'] = raw_date_text.replace('Đăng', '').strip() 
            
            if job_data['post_date']: print(f"  Ngày đăng (Post date) là: {job_data['post_date']}")
            else: print("  Không tìm thấy ngày đăng (Post date).") 
                
            if job_data['job_title'] and job_data['job_url']:
                all_jobs_data.append(job_data)
                print(f"  Đã thêm job item thứ {index + 1} vào danh sách.")
            else:
                print(f"  !! Job item thứ {index + 1} thiếu thông tin Title hoặc URL, không được thêm vào kết quả.")
                
        print(f'\n Đã xử lý xong {len(job_item_elements)} job items.') # Sửa lại, in ra số items đã duyệt qua
        if all_jobs_data:
            print(f"Tổng số jobs hợp lệ đã được trích xuất là: {len(all_jobs_data)}")
            print("\n Ví dụ 3 jobs đầu tiên đã trích xuất: ")
            for i, job_sample in enumerate(all_jobs_data[:3]):
                print(f"  --- Job {i + 1} ---")
                for key, value in job_sample.items(): # In tất cả các trường
                    print(f"    {key.replace('_', ' ').title()}: {value}")
        else:
            print("Không có job nào được trích xuất hợp lệ.")
            
    else:
        print("Không tìm thấy container 'div.box-job-list'. Vui lòng kiểm tra lại file HTML và selector.")

else:
    print("Biến 'page_source_html' không tồn tại hoặc rỗng. Hãy chạy lại Cell 3 để lấy HTML.")

Đang thực hiện phân tích (Parse) HTML với (kích thước là: 1440994 bytes) bằng BeautifulSoup...
Đã tìm thấy container 'div.box-job-list'.
Tìm thấy tổng cộng 50 job items trong 'div.box-job-list'.

 --- Đang xử lý job item thứ 1/50 ---
  Tên công việc (Job title) là: Nhân Viên Kỹ Thuật Máy Pha Cà Phê _ Lương Upto 15tr
  URL chi tiết công việc là: https://www.topcv.vn/viec-lam/nhan-vien-ky-thuat-may-pha-ca-phe-luong-upto-15tr/1725029.html?ta_source=JobSearchList_LinkDetail&u_sr_id=lCJNNkbcR4spKya9vcRSa3oHSuQ1GRLDRoGjqdzb_1747814448
  Tên công ty (Company name) là: CÔNG TY TNHH CÀ PHÊ KAFIN
  Mức lương (Salary) là: 7 - 13 triệu
  Địa chỉ (Location) là: Hồ Chí Minh
  Kinh nghiệm (Experience) là: Không yêu cầu
  Ngày đăng (Post date) là: 1 tuần trước
  Đã thêm job item thứ 1 vào danh sách.

 --- Đang xử lý job item thứ 2/50 ---
  Tên công việc (Job title) là: Nhân Viên Hỗ Trợ Khách Hàng Sau Giải Ngân /Nhân Viên Nhắc Phí - ( Thu Nhập 8-25 Triệu / Tháng - Thử...
  URL chi tiết công việc là: ht

# Cell 5: Save Extracted Data to JSON File and Close Webdriver
Mục đích: Lưu toàn bộ dữ liệu vào file JSON cuối cùng và quan trọng là đóng trình duyệt Selenium

In [6]:
# Kiểm tra xem biến all_jobs_data từ Cell 4 (cũ) có tồn tại và có dữ liệu không
if 'all_jobs_data' in locals() and all_jobs_data:
    try:
        # Sử dụng JSON_OUTPUT_FILENAME đã định nghĩa ở Cell 1
        print(f"\nĐang chuẩn bị lưu {len(all_jobs_data)} job items (từ trang 1) vào file JSON...")
        with open(JSON_OUTPUT_FILENAME, 'w', encoding='utf-8') as f_json:
            json.dump(all_jobs_data, f_json, ensure_ascii=False, indent=4)
        print(f"Thành công! Dữ liệu trang 1 đã được lưu vào file: '{JSON_OUTPUT_FILENAME}'")
        
        print("\nVí dụ 3 job đầu tiên đã trích xuất (từ trang 1):")
        for i, job_sample in enumerate(all_jobs_data[:3]):
            print(f"  --- Job {i+1} ---")
            for key, value in job_sample.items():
                print(f"    {key.replace('_', ' ').title()}: {value}")
                    
    except Exception as e_save:
        print(f"Lỗi khi lưu file JSON: {e_save}")
elif 'all_jobs_data' in locals() and not all_jobs_data:
     print("Không có dữ liệu job nào để lưu (danh sách all_jobs_data từ trang 1 rỗng).")
else:
    print("Biến 'all_jobs_data' không tồn tại. Hãy đảm bảo Cell 4 (cũ) đã chạy đúng.")

# Đóng WebDriver (Rất quan trọng!)
if 'driver' in locals() and driver is not None:
    try:
        print("\nĐang đóng WebDriver...")
        driver.quit()
        print("WebDriver đã được đóng thành công.")
        # Đặt lại biến driver thành None để tránh sử dụng lại ở cell khác nếu chạy lại
        # và cũng để cell này có thể chạy lại mà không báo lỗi driver đã đóng
        driver = None  
    except Exception as e_quit:
        print(f"Lỗi khi đóng WebDriver: {e_quit}")
else:
    print("\nWebDriver không tồn tại hoặc đã được đóng trước đó.")

print("\n--- HOÀN TẤT QUÁ TRÌNH SCRAPING TRANG 1 ---")


Đang chuẩn bị lưu 50 job items (từ trang 1) vào file JSON...
Thành công! Dữ liệu trang 1 đã được lưu vào file: 'topcv_jobs_data_20250521_150047.json'

Ví dụ 3 job đầu tiên đã trích xuất (từ trang 1):
  --- Job 1 ---
    Job Title: Nhân Viên Kỹ Thuật Máy Pha Cà Phê _ Lương Upto 15tr
    Job Url: https://www.topcv.vn/viec-lam/nhan-vien-ky-thuat-may-pha-ca-phe-luong-upto-15tr/1725029.html?ta_source=JobSearchList_LinkDetail&u_sr_id=lCJNNkbcR4spKya9vcRSa3oHSuQ1GRLDRoGjqdzb_1747814448
    Company Name: CÔNG TY TNHH CÀ PHÊ KAFIN
    Salary: 7 - 13 triệu
    Location: Hồ Chí Minh
    Experience: Không yêu cầu
    Post Date: 1 tuần trước
  --- Job 2 ---
    Job Title: Nhân Viên Hỗ Trợ Khách Hàng Sau Giải Ngân /Nhân Viên Nhắc Phí - ( Thu Nhập 8-25 Triệu / Tháng - Thử...
    Job Url: https://www.topcv.vn/brand/concentrixservices/tuyen-dung/nhan-vien-ho-tro-khach-hang-sau-giai-ngan-nhan-vien-nhac-phi-thu-nhap-8-25-trieu-thang-thu-viec-100-luong-ho-chi-minh-j1698139.html?ta_source=JobSearchList_Li

# Cell 6: Load Data from JSON and Initial Processing with Pandas